In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

%matplotlib inline

In [2]:
data ='https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'

In [3]:
df = pd.read_csv(data)
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
n = len(df) # Check numnber of rows in dataset.
print(f'total: {n} rows')

total: 10000 rows


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   model_year           10000 non-null  int64  
 1   origin               10000 non-null  object 
 2   fuel_type            10000 non-null  object 
 3   drivetrain           10000 non-null  object 
 4   num_doors            10000 non-null  int64  
 5   engine_displacement  10000 non-null  int64  
 6   num_cylinders        10000 non-null  int64  
 7   horsepower           9123 non-null   float64
 8   vehicle_weight       10000 non-null  int64  
 9   acceleration         9736 non-null   float64
 10  fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(3), int64(5), object(3)
memory usage: 859.5+ KB


In [6]:
df.describe() # Describe basic statistics

,model_year,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
count,10000.000000,10000.00000,10000.000000,10000.000000,9123.000000,10000.000000,9736.000000,10000.000000
mean,1999.549500,3.50400,2268.807000,6.035200,254.446016,4286.754000,18.888044,29.997700
std,14.207925,0.95209,183.447355,0.542578,22.844613,266.212491,1.279543,2.930245
min,1975.000000,2.00000,1590.000000,4.000000,171.000000,3320.000000,14.000000,19.800000
25%,1987.000000,3.00000,2150.000000,6.000000,239.000000,4110.000000,18.000000,28.000000
50%,2000.000000,4.00000,2270.000000,6.000000,254.000000,4280.000000,18.900000,30.000000
75%,2012.000000,4.00000,2390.000000,6.000000,270.000000,4470.000000,19.700000,31.900000
max,2024.000000,5.00000,3110.000000,8.000000,334.000000,5460.000000,23.800000,41.200000


In [7]:
df.dtypes

,0
model_year,int64
origin,object
fuel_type,object
drivetrain,object
num_doors,int64
engine_displacement,int64
num_cylinders,int64
horsepower,float64
vehicle_weight,int64
acceleration,float64


# Preparing the dataset
Preparation:

Fill missing values with zeros.

In [8]:
df.isnull().sum()

,0
model_year,0
origin,0
fuel_type,0
drivetrain,0
num_doors,0
engine_displacement,0
num_cylinders,0
horsepower,877
vehicle_weight,0
acceleration,264


In [9]:
numerical_features = list(df.dtypes[df.dtypes == 'float64'].index) # Store the names of int64 columns in numerical_features
df[numerical_features] = df[numerical_features].fillna(0.0) # Fill NA to missing value in numerical_features.

In [10]:
df.isnull().sum()

,0
model_year,0
origin,0
fuel_type,0
drivetrain,0
num_doors,0
engine_displacement,0
num_cylinders,0
horsepower,0
vehicle_weight,0
acceleration,0


Split the data

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

In [12]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [13]:
target = 'fuel_efficiency_mpg'

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

df_train = df_train.drop(columns=[target])
df_val = df_val.drop(columns=[target])
df_test = df_test.drop(columns=[target])

In [14]:
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')
test_dicts = df_test.to_dict(orient='records')

In [15]:
dv = DictVectorizer(sparse=True)

In [16]:
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

In [17]:
print('Train:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

Train: (6000, 16)
Validation: (2000, 16)
Test: (2000, 16)


# Question 1
Train a decision tree regressor to predict the fuel_efficiency_mpg variable.

Train a model with max_depth=1.
Which original feature is used for splitting the data? If you use DictVectorizer, map a one-hot feature such as origin=USA back to its original column name.

In [18]:
from sklearn.tree import DecisionTreeRegressor, export_text

dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

# Feature used to split the data at the root node
feature_names = dv.get_feature_names_out()
split_feature = feature_names[dt.tree_.feature[0]]

# Map the one-hot feature name back to the original column name
original_feature = split_feature.split('=')[0]

print('Answer:', original_feature)
print(export_text(dt, feature_names=list(feature_names)))

Answer: model_year
|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



# Question 2
Train a random forest regressor with these parameters:

* n_estimators=10
* random_state=1
* n_jobs=-1 (optional - to make training faster)

What's the RMSE of this model on the validation data?

In [19]:
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(f'Validation RMSE: {rmse:.3f}')

Validation RMSE: 1.837


# Question 3
Now let's experiment with the n_estimators parameter

* Try these values of this parameter: [10, 50, 100, 150].
* Set random_state to 1.
* Evaluate the model on the validation dataset.

Which value gives the lowest validation RMSE? Consider 3 decimal places for calculating the answer.

In [20]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

scores = []

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    scores.append((n, round(rmse, 3)))

df_scores = pd.DataFrame(
    scores,
    columns=['n_estimators', 'validation_rmse']
)

print(df_scores.to_string(index=False))

best = df_scores[
    df_scores.validation_rmse == df_scores.validation_rmse.min()
]

print('\nLowest validation RMSE:')
print(best.to_string(index=False))

 n_estimators  validation_rmse
           10            1.837
           50            1.772
          100            1.768
          150            1.769

Lowest validation RMSE:
 n_estimators  validation_rmse
          100            1.768


#Question 4
Let's select the best max_depth:

* Try different values of max_depth: [10, 15, 20, 25]
* For each of these values,
    *   try different values of n_estimators from [10, 50, 100, 150]
    *   calculate the mean RMSE
* Fix the random seed: random_state=1

What's the best max_depth, using the mean RMSE?

In [21]:
scores = []

for depth in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(
            max_depth=depth,
            n_estimators=n,
            random_state=1,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        scores.append((depth, n, rmse))

df_scores = pd.DataFrame(
    scores,
    columns=['max_depth', 'n_estimators', 'rmse']
)

# Calculate the mean using unrounded scores before rounding
mean_rmse = df_scores.groupby('max_depth')['rmse'].mean()

print(mean_rmse.round(3))
print('\nBest max_depth:', mean_rmse.idxmin())

max_depth
10    1.755
15    1.784
20    1.788
25    1.786
Name: rmse, dtype: float64

Best max_depth: 10


# Question 5
At each step of the decision tree learning algorithm, it finds the best split. When doing it, we can calculate "gain" - the reduction in impurity before and after the split. This gain is quite useful in understanding what are the important features for tree-based models.

we'll find the most important feature:

Train the model with these parameters:
* n_estimators=10,
* max_depth=20,
* random_state=1,
* n_jobs=-1 (optional)

Get the feature importance information from this model
What's the most important feature (among these 4)?

In [22]:
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

importance = pd.Series(
    rf.feature_importances_,
    index=dv.get_feature_names_out()
)

# Aggregate one-hot feature importances by original column
importance.index = importance.index.str.split('=').str[0]
importance = importance.groupby(level=0).sum()
importance = importance.sort_values(ascending=False)

print(importance.to_string())

model_year             0.296001
vehicle_weight         0.203879
fuel_type              0.193551
horsepower             0.078017
engine_displacement    0.061005
acceleration           0.055201
origin                 0.052040
drivetrain             0.033608
num_doors              0.019415
num_cylinders          0.007284


# Question 6
Now let's train an XGBoost model! For this question, we'll tune the eta parameter:

* Install XGBoost
* Create DMatrix for train and validation
* Create a watchlist
* Train a model with these parameters for 100 rounds:

Now change eta from 0.3 to 0.1.

Which eta leads to the best RMSE score on the validation dataset?

In [23]:
%pip install xgboost

In [24]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

watchlist = [(dtrain, 'train'), (dval, 'val')]

scores = []

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }

    model = xgb.train(
        params=xgb_params,
        dtrain=dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=False
    )

    y_pred = model.predict(dval)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    scores.append((eta, rmse))

results = pd.DataFrame(scores, columns=['eta', 'validation_rmse'])

print(results.to_string(index=False))
print('\nAnswer eta:', results.loc[results.validation_rmse.idxmin(), 'eta'])

 eta  validation_rmse
 0.3         1.826458
 0.1         1.724810

Answer eta: 0.1
